# 02 - Benchmark de Modelos

Neste notebook, vamos comparar a performance de 4 modelos na tarefa de classificação do dataset olist:
1. BERTugues
2. BERTimbau Base
3. BERTimbau Large
4. mBERT

Vamos extrair os hidden states (CLS) e treinar um RandomForestClassifier para cada um.


In [1]:
import pandas as pd
import numpy as np
import time
from datasets import Dataset
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
df = pd.read_csv('datasets/imdb-reviews-pt-br.csv', index_col="id")
df['y'] = df['sentiment'].apply(lambda x: 1 if x == "pos" else 0)
df.drop("text_en", axis = 1, inplace=True)
dataset = Dataset.from_pandas(df)

In [3]:
df

,text_pt,sentiment,y
id,,,
1,"Mais uma vez, o Sr. Costner arrumou um filme p...",neg,0
2,Este é um exemplo do motivo pelo qual a maiori...,neg,0
3,"Primeiro de tudo eu odeio esses raps imbecis, ...",neg,0
4,Nem mesmo os Beatles puderam escrever músicas ...,neg,0
5,Filmes de fotos de latão não é uma palavra apr...,neg,0
...,...,...,...
49456,"Como a média de votos era muito baixa, e o fat...",pos,1
49457,O enredo teve algumas reviravoltas infelizes e...,pos,1
49458,Estou espantado com a forma como este filme e ...,pos,1


In [4]:
modelos = {
    "BERTugues": "ricardoz/BERTugues-base-portuguese-cased",
    "BERTimbau Base": "neuralmind/bert-base-portuguese-cased",
    "BERTimbau Large": "neuralmind/bert-large-portuguese-cased",
    "mBERT": "bert-base-multilingual-cased"
}

resultados = []

In [5]:
for nome_modelo, caminho_modelo in modelos.items():
    print(f"\n{'='*50}")
    print(f"Avaliando modelo: {nome_modelo}")
    print(f"{'='*50}")
    
    # 1. Carregar Tokenizer e Modelo
    tokenizer = AutoTokenizer.from_pretrained(caminho_modelo, do_lower_case=False)
    model = AutoModel.from_pretrained(caminho_modelo).to(device)
    
    # 2. Tokenização
    def tokenize(batch):
        return tokenizer(batch["text_pt"], padding=True, truncation=True, max_length=512)
    
    print("Tokenizando...")
    encoded = dataset.map(tokenize, batched=True, batch_size=None)
    encoded.set_format("torch", columns=["input_ids", "attention_mask"])
    
    # 3. Extração de Hidden States
    def extract_hidden_states(batch):
        inputs = {k: v.to(device) for k, v in batch.items() if k in tokenizer.model_input_names}
        with torch.no_grad():
            last_hidden_state = model(**inputs).last_hidden_state
        return {"hidden_state": last_hidden_state[:, 0].cpu().numpy()} # [CLS] token
    
    print("Extraindo Hidden States...")
    t0 = time.time()
    # Batch size em 64 ou menor para suportar o model Large em GPUs normais
    encoded_hidden = encoded.map(extract_hidden_states, batched=True, batch_size=32) 
    t_extracao = time.time() - t0
    
    # Limpar da memória RAM/VRAM
    del model
    torch.cuda.empty_cache()
    
    # 4. Preparar Dados para o Scikit-Learn
    df_hidden = encoded_hidden.to_pandas()
    X = np.stack(df_hidden["hidden_state"].values)
    y = df_hidden["y"].values.astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 5. Treinar Random Forest
    print("Treinando Random Forest...")
    clf = RandomForestClassifier(max_depth=50, n_estimators=500, random_state=0, n_jobs=6)
    
    t0 = time.time()
    clf.fit(X_train, y_train)
    t_treino = time.time() - t0
    
    # 6. Avaliar
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"\n> Acurácia {nome_modelo}: {acc:.4f} | F1: {f1:.4f}")
    print(f"> Tempo de Extração: {t_extracao:.2f}s | Tempo de Treino: {t_treino:.2f}s")
    
    resultados.append({
        "Modelo": nome_modelo,
        "Acurácia": acc,
        "F1": f1,
        "Tempo Extração (s)": round(t_extracao, 2),
        "Tempo Treino RF (s)": round(t_treino, 2)
    })


Avaliando modelo: BERTugues


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizando...


Map:   0%|          | 0/49459 [00:00<?, ? examples/s]

...

Tokenizando...


Map:   0%|          | 0/49459 [00:00<?, ? examples/s]

Extraindo Hidden States...


Map:   0%|          | 0/49459 [00:00<?, ? examples/s]

Treinando Random Forest...

> Acurácia mBERT: 0.7183 | F1: 0.7183
> Tempo de Extração: 371.10s | Tempo de Treino: 78.76s


In [6]:
# Tabela Final de Resultados
df_resultados = pd.DataFrame(resultados)
df_resultados.sort_values(by="Acurácia", ascending=False, inplace=True)
df_resultados.reset_index(drop=True, inplace=True)
df_resultados

,Modelo,Acurácia,F1,Tempo Extração (s),Tempo Treino RF (s)
0,BERTimbau Large,0.854428,0.850715,1245.52,98.55
1,BERTugues,0.846442,0.841490,447.05,101.45
2,BERTimbau Base,0.824808,0.823362,446.33,92.65
3,mBERT,0.718257,0.718343,371.10,78.76
